# ParkCast SF — Inference-support parquet exports

The training CSV joined ~6 data pipelines, but only 5 of them got persisted as inference-ready lookups (`block_aggs`, `blocks`, `lag_history`, `citations_hourly_median`, `sfpark_calibration`). Features from the other joins (POI counts, 311 complaints, permits, raw citation counts) only existed as columns in the training CSV and never got written to serving-side files.

This notebook fixes that by exporting a per-block **static features** parquet covering the 9 features that had no lookup:

- `poi_dining_200m, poi_retail_200m, poi_transit_200m, poi_attraction_200m` — POI counts within 200m (truly static per block → `first`)
- `permit_active, permit_count_30d` — permit state (time-varying → `last`, i.e. most recent snapshot)
- `citation_count, complaints_311_median, complaints_311_total` — per-block historical aggregates (→ `median` / `last`)

After running this, `app/main.py` can cover the full 33-feature model input without NaN fallbacks.

In [1]:
import os
import pandas as pd

PROJECT_DIR = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(PROJECT_DIR, "data", "processed_training_data.csv")
OUT_PATH = os.path.join(PROJECT_DIR, "app", "models", "block_static_features.parquet")

In [2]:
print("Loading training CSV...")
cols = [
    "lat", "lon", "timestamp", "target_is_estimated",
    "citation_count", "complaints_311_median", "complaints_311_total",
    "poi_dining_200m", "poi_retail_200m", "poi_transit_200m", "poi_attraction_200m",
    "permit_active", "permit_count_30d",
    "sfpark_block_occ",
]
df = pd.read_csv(DATA_PATH, usecols=cols, parse_dates=["timestamp"])
# Match the training filter exactly so block-level aggregates reflect the
# same rows the model saw.
df = df[df["target_is_estimated"] == 0].drop(columns=["target_is_estimated"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"  rows: {len(df):,}   unique blocks: {df[['lat','lon']].drop_duplicates().shape[0]:,}")

Loading training CSV...


  rows: 5,637,410   unique blocks: 1,413


In [3]:
# Per-block aggregation strategy per feature:
#   POI counts — static per block, take the first value seen.
#   permit_active / permit_count_30d — most recent snapshot (sorted → last).
#   citation_count / complaints_311_* / sfpark_block_occ — historical "typical"
#     value per block; median for rate-like series, last for totals.
agg_spec = {
    "poi_dining_200m":       "first",
    "poi_retail_200m":       "first",
    "poi_transit_200m":      "first",
    "poi_attraction_200m":   "first",
    "permit_active":         "last",
    "permit_count_30d":      "last",
    "citation_count":        "median",
    "complaints_311_median": "median",
    "complaints_311_total":  "last",
    "sfpark_block_occ":      "median",
}
static = df.groupby(["lat", "lon"], sort=False).agg(agg_spec).reset_index()
print(static.head())
print(f"\nblocks: {len(static):,}")

         lat         lon  poi_dining_200m  poi_retail_200m  poi_transit_200m  \
0  37.783100 -122.459485               12               26                 0   
1  37.807611 -122.421104               14               13                 0   
2  37.764375 -122.420616               41               38                 4   
3  37.738567 -122.468537               15               34                 0   
4  37.752903 -122.420628               24               42                 1   

   poi_attraction_200m  permit_active  permit_count_30d  citation_count  \
0                    0              1                 1             0.0   
1                    8              0                 0             0.0   
2                    8              0                 0             0.0   
3                    2              0                 0             0.0   
4                    2              0                 0             0.0   

   complaints_311_median  complaints_311_total  sfpark_block_occ  
0

In [4]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
static.to_parquet(OUT_PATH, index=False)
print(f"Wrote: {OUT_PATH}  ({os.path.getsize(OUT_PATH)/1024:.0f} KB)")

Wrote: /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/block_static_features.parquet  (41 KB)
